# Results analysis - tables and charts for the report

Reads every `results/*.json` written by `pipeline.ipynb`. No Spark needed.

1. Overview of every run
2. Per-stage breakdown and skew of one run
3. Before/after comparison of two runs (one per optimisation)
4. Optimisation summary table (report section 4)
5. Dataset scale chart and cluster-size speedup chart (report section 5)

In [1]:
import glob
import json
import os

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = "../results"
FIGURES_DIR = "../report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Chart styling: one measured series (blue), reference lines in muted grey.
BLUE, GREY, INK = "#2a78d6", "#52514e", "#0b0b0b"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GREY, "axes.labelcolor": INK, "xtick.color": GREY, "ytick.color": GREY,
    "axes.grid": True, "grid.color": "#e5e4df", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "lines.markersize": 8, "font.size": 10,
})


def load_results(results_dir=RESULTS_DIR):
    runs = {}
    for path in sorted(glob.glob(os.path.join(results_dir, "*.json"))):
        with open(path) as f:
            data = json.load(f)
        runs[data["metadata"]["run_name"]] = data
    return runs


def fmt_bytes(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}PB"


results = load_results()
print(f"{len(results)} runs:", list(results))

45 runs: ['cluster-3w-baseline-r1', 'cluster-3w-baseline-r2', 'cluster-3w-baseline-r3', 'cluster-all-opts-r1', 'cluster-all-opts-r2', 'cluster-all-opts-r3', 'cluster-control-r1', 'cluster-control-r2', 'cluster-control-r3', 'cluster-opt1+2-r1', 'cluster-opt1+2-r2', 'cluster-opt1+2-r3', 'cluster-opt1-cache-input-r1', 'cluster-opt1-cache-input-r2', 'cluster-opt1-cache-input-r3', 'cluster-opt2-cache-intermediate-r1', 'cluster-opt2-cache-intermediate-r2', 'cluster-opt2-cache-intermediate-r3', 'local-all-opts-r1', 'local-all-opts-r2', 'local-all-opts-r3', 'local-baseline-r1', 'local-baseline-r2', 'local-baseline-r3', 'local-control-r1', 'local-control-r2', 'local-control-r3', 'local-opt1+2-r1', 'local-opt1+2-r2', 'local-opt1+2-r3', 'local-opt1-cache-input-r1', 'local-opt1-cache-input-r2', 'local-opt1-cache-input-r3', 'local-opt2-cache-intermediate-r1', 'local-opt2-cache-intermediate-r2', 'local-opt2-cache-intermediate-r3', 'local-opt3-split16-r1', 'local-opt3-split16-r2', 'local-opt3-split16

## 1. Overview of every run

In [2]:
def overview(results):
    rows = []
    for name, data in results.items():
        m = data["metadata"]
        row = {"run": name, "master": m["master"], "executors": m.get("num_executors"),
               "cores": m.get("total_cores"), "years": ",".join(map(str, m["years"])),
               "fraction": m.get("fraction"), "total_s": m["total_wall_time_s"]}
        row.update({f"{op}_s": r["wall_time_s"] for op, r in data["runs"].items()})
        rows.append(row)
    return pd.DataFrame(rows).set_index("run")


overview(results)

,master,executors,cores,years,fraction,total_s,m1_s,m2_s,m3_s,o2_s,o3_s
run,,,,,,,,,,,
cluster-3w-baseline-r1,spark://spark-master:7077,3,6,2024,None,153.49,9.87,18.21,18.52,20.04,86.85
cluster-3w-baseline-r2,spark://spark-master:7077,3,6,2024,None,121.14,10.76,12.24,10.49,15.12,72.53
cluster-3w-baseline-r3,spark://spark-master:7077,3,6,2024,None,111.02,9.63,13.87,16.55,11.46,59.51
cluster-all-opts-r1,spark://spark-master:7077,3,6,2024,None,44.44,14.25,6.02,4.85,12.01,7.31
cluster-all-opts-r2,spark://spark-master:7077,3,6,2024,None,38.67,13.62,5.87,4.16,9.59,5.43
cluster-all-opts-r3,spark://spark-master:7077,3,6,2024,None,37.06,11.69,4.07,4.10,9.62,7.58
cluster-control-r1,spark://spark-master:7077,3,6,2024,None,145.78,9.71,14.90,13.61,11.44,96.12
cluster-control-r2,spark://spark-master:7077,3,6,2024,None,95.75,6.52,10.25,8.44,14.90,55.64
cluster-control-r3,spark://spark-master:7077,3,6,2024,None,82.01,7.61,10.01,8.43,10.58,45.38


## 2. Per-operation totals and per-stage breakdown of one run

The brief asks for shuffle read/write bytes and task counts (on your
machine) and wall-clock and per-stage duration (on the cluster).
`skew_max_over_median` ≥ 10 marks a stage where one task takes 10× the
median.

In [3]:
BYTE_COLS = ["input_bytes", "output_bytes", "shuffle_read_bytes", "shuffle_write_bytes",
             "memory_spilled_bytes", "disk_spilled_bytes"]


def op_table(run):
    df = pd.DataFrame({op: {k: v for k, v in r.items() if k != "stages"}
                       for op, r in results[run]["runs"].items()}).T
    for c in BYTE_COLS:
        df[c] = df[c].map(fmt_bytes)
    return df


def stage_table(run, op):
    df = pd.DataFrame(results[run]["runs"][op]["stages"]).set_index("stage_id")
    df["skewed"] = df["skew_max_over_median"].fillna(0) >= 10
    for c in [c for c in BYTE_COLS if c in df]:
        df[c] = df[c].map(fmt_bytes)
    return df.sort_values("duration_ms", ascending=False)


RUN = next(iter(results), None)   # pick the run to inspect
if RUN:
    display(op_table(RUN))

,wall_time_s,num_jobs,num_stages,num_tasks,executor_run_time_ms,executor_cpu_time_ms,input_bytes,output_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,input_records,jvm_gc_time_ms,shuffle_fetch_wait_ms,shuffle_write_time_ms,shuffle_remote_read_bytes
m1,9.87,4.0,4.0,75.0,45979.0,32618.0,514.6MB,0.0B,8.9KB,4.5KB,0.0B,0.0B,41169720.0,2106.0,0.0,478.0,5928.0
m2,18.21,3.0,3.0,75.0,94769.0,80086.0,514.7MB,52.2KB,2.2MB,2.2MB,0.0B,0.0B,41169985.0,4440.0,0.0,1149.0,1530518.0
m3,18.52,7.0,7.0,78.0,93543.0,82453.0,514.7MB,177.5KB,2.2MB,2.2MB,0.0B,0.0B,41169985.0,3702.0,0.0,872.0,1009055.0
o2,20.04,15.0,15.0,122.0,103543.0,86634.0,514.7MB,6.3MB,108.5MB,86.1MB,0.0B,0.0B,41170515.0,3812.0,15.0,1862.0,43726337.0
o3,86.85,14.0,14.0,416.0,489038.0,442627.0,2.5GB,7.2MB,116.8MB,116.8MB,0.0B,0.0B,205848865.0,20307.0,19.0,4320.0,76396768.0


In [4]:
if RUN:
    for op in results[RUN]["runs"]:
        print(op)
        display(stage_table(RUN, op).head(10))

m1


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,executor_run_time_ms,jvm_gc_time_ms,shuffle_fetch_wait_ms,peak_execution_memory,skewed
stage_id,,,,,,,,,,,,,,,,
12,collect at /tmp/ipykernel_784/2181839698.py:4,8699,72,576.0,2883.0,5.01,514.6MB,0.0B,4.4KB,0.0B,0.0B,45831,2106,0,18874368,False
16,collect at /tmp/ipykernel_784/2181839698.py:4,140,1,66.0,66.0,1.00,0.0B,4.4KB,107.0B,0.0B,0.0B,66,0,0,17039344,False
14,collect at /tmp/ipykernel_784/2181839698.py:4,134,1,76.0,76.0,1.00,0.0B,4.4KB,0.0B,0.0B,0.0B,76,0,0,17039344,False
19,collect at /tmp/ipykernel_784/2181839698.py:4,35,1,6.0,6.0,1.00,0.0B,107.0B,0.0B,0.0B,0.0B,6,0,0,0,False


m2


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,executor_run_time_ms,jvm_gc_time_ms,shuffle_fetch_wait_ms,peak_execution_memory,skewed
stage_id,,,,,,,,,,,,,,,,
21,$anonfun$withThreadLocalCaptured$2 at Completa...,16518,72,1386.0,3990.0,2.88,514.6MB,0.0B,2.2MB,0.0B,0.0B,93875,4440,0,758038272,False
23,$anonfun$withThreadLocalCaptured$2 at Completa...,546,2,423.0,423.0,1.00,0.0B,2.2MB,0.0B,0.0B,0.0B,827,0,0,67764160,False
20,$anonfun$withThreadLocalCaptured$2 at Completa...,93,1,67.0,67.0,1.00,12.0KB,0.0B,0.0B,0.0B,0.0B,67,0,0,0,False


m3


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,executor_run_time_ms,jvm_gc_time_ms,shuffle_fetch_wait_ms,peak_execution_memory,skewed
stage_id,,,,,,,,,,,,,,,,
25,$anonfun$withThreadLocalCaptured$2 at Completa...,16421,72,1426.0,3450.0,2.42,514.6MB,0.0B,1.5MB,0.0B,0.0B,92822,3702,0,20361024,False
45,$anonfun$withThreadLocalCaptured$2 at Completa...,254,1,171.0,171.0,1.00,0.0B,192.3KB,0.0B,0.0B,0.0B,171,0,0,16908272,False
34,$anonfun$withThreadLocalCaptured$2 at Completa...,235,1,211.0,211.0,1.00,0.0B,165.7KB,158.5KB,0.0B,0.0B,211,0,0,16908272,False
39,$anonfun$withThreadLocalCaptured$2 at Completa...,180,1,123.0,123.0,1.00,0.0B,158.5KB,192.3KB,0.0B,0.0B,123,0,0,16908272,False
30,$anonfun$withThreadLocalCaptured$2 at Completa...,120,1,78.0,78.0,1.00,0.0B,202.5KB,165.7KB,0.0B,0.0B,78,0,0,16908272,False
27,$anonfun$withThreadLocalCaptured$2 at Completa...,99,1,73.0,73.0,1.00,0.0B,1.5MB,202.5KB,0.0B,0.0B,73,0,0,17039344,False
24,$anonfun$withThreadLocalCaptured$2 at Completa...,82,1,65.0,65.0,1.00,12.0KB,0.0B,0.0B,0.0B,0.0B,65,0,0,0,False


o2


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,executor_run_time_ms,jvm_gc_time_ms,shuffle_fetch_wait_ms,peak_execution_memory,skewed
stage_id,,,,,,,,,,,,,,,,
49,$anonfun$withThreadLocalCaptured$2 at Completa...,14902,72,1249.0,3392.0,2.72,514.6MB,0.0B,11.2MB,0.0B,0.0B,85419,3579,0,757071168,False
55,$anonfun$withThreadLocalCaptured$2 at Completa...,1601,6,939.0,1061.0,1.13,0.0B,18.0MB,24.3MB,0.0B,0.0B,4827,94,0,113246112,False
59,$anonfun$withThreadLocalCaptured$2 at Completa...,1416,4,474.0,546.0,1.15,0.0B,4.2MB,2.9MB,0.0B,0.0B,1910,16,0,307756800,False
53,$anonfun$withThreadLocalCaptured$2 at Completa...,1036,6,264.0,298.0,1.13,0.0B,11.2MB,4.2MB,0.0B,0.0B,1471,36,1,113347872,False
79,$anonfun$withThreadLocalCaptured$2 at Completa...,845,6,636.0,803.0,1.26,0.0B,21.1MB,0.0B,0.0B,0.0B,4067,42,0,113246112,False
52,$anonfun$withThreadLocalCaptured$2 at Completa...,781,6,295.0,327.0,1.11,0.0B,11.2MB,3.3MB,0.0B,0.0B,1735,0,0,217186512,False
57,$anonfun$withThreadLocalCaptured$2 at Completa...,691,3,115.0,144.0,1.25,0.0B,3.3MB,5.7KB,0.0B,0.0B,308,0,0,104333216,False
62,$anonfun$withThreadLocalCaptured$2 at Completa...,439,1,32.0,32.0,1.00,0.0B,5.7KB,0.0B,0.0B,0.0B,32,0,0,16842736,False
51,$anonfun$withThreadLocalCaptured$2 at Completa...,419,6,327.0,347.0,1.06,0.0B,11.2MB,18.0MB,0.0B,0.0B,1893,20,14,113569728,False


o3


,name,duration_ms,num_tasks,task_median_ms,task_max_ms,skew_max_over_median,input_bytes,shuffle_read_bytes,shuffle_write_bytes,memory_spilled_bytes,disk_spilled_bytes,executor_run_time_ms,jvm_gc_time_ms,shuffle_fetch_wait_ms,peak_execution_memory,skewed
stage_id,,,,,,,,,,,,,,,,
85,$anonfun$withThreadLocalCaptured$2 at Completa...,81089,72,2346.0,4969.0,2.12,514.6MB,0.0B,20.2MB,0.0B,0.0B,134808,5873,0,725351776,False
84,$anonfun$withThreadLocalCaptured$2 at Completa...,59686,72,1480.0,3361.0,2.27,514.6MB,0.0B,21.9MB,0.0B,0.0B,87671,2935,0,759168320,False
87,$anonfun$withThreadLocalCaptured$2 at Completa...,55504,6,116.0,496.0,4.28,0.0B,21.9MB,3.7MB,0.0B,0.0B,1076,28,0,253230912,False
89,$anonfun$withThreadLocalCaptured$2 at Completa...,44738,1,49.0,49.0,1.00,0.0B,44.2KB,441.0B,0.0B,0.0B,49,0,0,524288,False
83,$anonfun$withThreadLocalCaptured$2 at Completa...,43785,72,718.0,1494.0,2.08,514.6MB,0.0B,4.1KB,0.0B,0.0B,42771,1768,0,0,False
91,$anonfun$withThreadLocalCaptured$2 at Completa...,36902,1,30.0,30.0,1.00,0.0B,4.1KB,0.0B,0.0B,0.0B,30,0,9,0,False
82,$anonfun$withThreadLocalCaptured$2 at Completa...,36350,72,990.0,3901.0,3.94,514.6MB,0.0B,44.2KB,0.0B,0.0B,69216,2865,0,18874368,False
81,$anonfun$withThreadLocalCaptured$2 at Completa...,25433,72,2018.0,5892.0,2.92,514.6MB,0.0B,21.9MB,0.0B,0.0B,141638,6077,0,759168320,False
93,$anonfun$withThreadLocalCaptured$2 at Completa...,20386,6,303.0,414.0,1.37,0.0B,21.9MB,16.3MB,0.0B,0.0B,1735,17,0,150994848,False


## 3. Before/after comparison

Compare two runs, either for one operation or for the whole pipeline
(`op=None`). Each optimisation in the report needs this "before"
evidence, not just the "after".

In [5]:
METRICS = ["wall_time_s", "num_stages", "num_tasks", "executor_run_time_ms",
           "executor_cpu_time_ms"] + BYTE_COLS


def totals(run, op=None):
    runs = results[run]["runs"]
    selected = [runs[op]] if op else list(runs.values())
    return {k: sum(r[k] for r in selected) for k in METRICS}


def compare(before, after, op=None):
    b, a = totals(before, op), totals(after, op)
    rows = []
    for k in METRICS:
        change = f"{(a[k] - b[k]) / b[k] * 100:+.1f}%" if b[k] else "-"
        show = fmt_bytes if k in BYTE_COLS else (lambda v: v)
        rows.append({"metric": k, before: show(b[k]), after: show(a[k]), "change": change})
    return pd.DataFrame(rows).set_index("metric")


# Example: compare("baseline", "opt-broadcast", op="o2")

## 4. Optimisation summary table (report section 4)

Each configuration ran 3 times next to a control run of the baseline in the same session
(`<env>-control-r1..r3`). The table shows the medians and the speedup against that control.

In [ ]:
import statistics


def reps(prefix):
    """The repetitions <prefix>-r1, -r2, ... of one configuration."""
    return [k for k in results if k.rsplit("-r", 1)[0] == prefix]


def median_total(prefix):
    runs = reps(prefix)
    return statistics.median(results[k]["metadata"]["total_wall_time_s"] for k in runs) if runs else None


OPTIMISATIONS = [
    ("1. Reading the input once", "opt1-cache-input"),
    ("2. Caching O3's hourly", "opt2-cache-intermediate"),
    ("3. Read splits = row groups", "opt3-split16"),
    ("4. Aligned self-join keys", "opt4-align-keys"),
    ("5. Broadcast hints", "opt5-broadcast"),
    ("1 + 2", "opt1+2"),
    ("All combined", "all-opts"),
]

rows = []
for title, label in OPTIMISATIONS:
    row = {"optimisation": title}
    for env in ("local", "cluster"):
        before, after = median_total(f"{env}-control"), median_total(f"{env}-{label}")
        row[f"{env}_before_s"] = before
        row[f"{env}_after_s"] = after
        row[f"{env}_speedup"] = round(before / after, 2) if before and after else None
    rows.append(row)
summary = pd.DataFrame(rows).set_index("optimisation")
summary

## 5a. Dataset scale

The optimised pipeline (all five switches) on the cluster, on real subsets of the data:
1, 2, 3, 6 and 12 months of 2024, and 2024+2025 (`cluster-scale-<n>m-r*`). The 12-month
point reuses `cluster-all-opts-r1..r3` (same configuration); the other sizes have one run each.
Subsets are used instead of `FRACTION`, because sampling happens after reading, so
the files would still be read completely. The dashed line is linear scaling through the
full-year point: points above it scale worse than linear, points below it better.

In [ ]:
import pyarrow.parquet as pq

DATA_DIR = "../data"


def trips_in(run):
    """Number of trips in the input files of a run, from the Parquet footers."""
    m = results[run]["metadata"]
    return sum(pq.ParquetFile(os.path.join(DATA_DIR, f"yellow_tripdata_{y}-{mo:02d}.parquet")).metadata.num_rows
               for y in m["years"] for mo in m["months"])


rows = []
for months in (1, 2, 3, 6, 12, 24):
    # the full year on 3 workers is exactly the configuration of cluster-all-opts
    runs = reps(f"cluster-scale-{months}m") or (reps("cluster-all-opts") if months == 12 else [])
    if runs:
        t = [results[k]["metadata"]["total_wall_time_s"] for k in runs]
        rows.append({"months": months, "trips_M": trips_in(runs[0]) / 1e6,
                     "time_s": statistics.median(t), "min_s": min(t), "max_s": max(t)})
scale = pd.DataFrame(rows)

if len(scale) and (scale.months == 12).any():   # the full year is the reference point
    full = scale[scale.months == 12].iloc[0]
    scale["share_of_year"] = scale.trips_M / full.trips_M
    scale["time_vs_linear"] = scale.time_s / (full.time_s * scale.share_of_year)
    scale["s_per_M_trips"] = scale.time_s / scale.trips_M
    display(scale.round(2))

    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    x_max = scale.trips_M.max() * 1.05
    ax.plot([0, x_max], [0, full.time_s * x_max / full.trips_M], "--", color=GREY,
            linewidth=1.5, label="linear (through 1 year)")
    ax.errorbar(scale.trips_M, scale.time_s,
                yerr=[scale.time_s - scale.min_s, scale.max_s - scale.time_s],
                fmt="-o", color=BLUE, ecolor=BLUE, elinewidth=1, capsize=3, label="measured (median, min-max)")
    for _, r in scale[scale.months.isin([1, 12, 24])].iterrows():
        label = {1: "1 month", 12: "1 year", 24: "2 years"}[r.months]
        ax.annotate(label, (r.trips_M, r.time_s), textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=INK)
    ax.set_xlabel("Trips in the input (millions)")
    ax.set_ylabel("Wall-clock time (s)")
    ax.set_xlim(0, x_max)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "scale_fraction.pdf"), bbox_inches="tight")
    plt.show()

## 5b. Cluster size: speedup and efficiency

The optimised pipeline on the full year of 2024 with 1-5 workers of 2 cores each
(`cluster-workers-<n>w-r*`). The 3-worker point reuses
`cluster-all-opts-r1..r3` (same configuration); the other worker counts have one run each. Speedup is relative to 1 worker:
S(n) = T(1) / T(n), efficiency E(n) = S(n) / n. All workers run on the same 10-core machine.

In [ ]:
rows = []
for n in range(1, 6):
    # 3 workers on the full year is exactly the configuration of cluster-all-opts
    runs = reps(f"cluster-workers-{n}w") or (reps("cluster-all-opts") if n == 3 else [])
    if runs:
        t = [results[k]["metadata"]["total_wall_time_s"] for k in runs]
        rows.append({"workers": n, "cores": results[runs[0]]["metadata"]["total_cores"],
                     "time_s": statistics.median(t), "min_s": min(t), "max_s": max(t)})
work = pd.DataFrame(rows)

if len(work):
    t1 = work.time_s.iloc[0]
    work["speedup"] = t1 / work.time_s
    work["efficiency"] = work.speedup / (work.workers / work.workers.iloc[0])
    display(work.round(2))

    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.plot(work.workers, work.workers / work.workers.iloc[0], "--", color=GREY,
            linewidth=1.5, label="ideal (linear)")
    ax.errorbar(work.workers, work.speedup,
                yerr=[work.speedup - t1 / work.max_s, t1 / work.min_s - work.speedup],
                fmt="-o", color=BLUE, ecolor=BLUE, elinewidth=1, capsize=3, label="measured (median, min-max)")
    for _, r in work.iterrows():
        ax.annotate(f"E={r.efficiency:.2f}", (r.workers, r.speedup), textcoords="offset points",
                    xytext=(0, -14), ha="center", fontsize=8, color=INK)
    ax.set_xlabel("Workers (2 cores each)")
    ax.set_ylabel("Speedup vs. 1 worker")
    ax.set_xticks(work.workers)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, "speedup_workers.pdf"), bbox_inches="tight")
    plt.show()